# Notebook 4: Out-of-Sample Backtest & Quant Performance Report

Trong notebook này, chúng ta thực hiện kiểm định (Backtest) chiến lược trên dữ liệu hoàn toàn chưa biết (Out-of-Sample từ `2023-07-01` đến `2024-06-30`):
1. Mô phỏng tái cơ cấu danh mục hàng tuần/hàng kỳ với chi phí giao dịch và thuế (`0.20%/trade`).
2. Tính toán bảng KPI tài chính định lượng chuyên nghiệp: **Sharpe Ratio**, **Maximum Drawdown**, **CAGR**, **Information Ratio**.
3. Trực quan hóa biểu đồ chuẩn xuất bản: Equity Curve, Underwater Drawdowns, Rolling Beta, và Allocation History.


In [ ]:
!git clone https://github.com/PTN2004/AI-Driven-Market-Neutral-Portfolio-Optimization.git
%cd AI-Driven-Market-Neutral-Portfolio-Optimization
!pip install -qr requirements.txt

Cloning into 'AI-Driven-Market-Neutral-Portfolio-Optimization'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 221 (delta 93), reused 214 (delta 86), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 4.54 MiB | 8.65 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/AI-Driven-Market-Neutral-Portfolio-Optimization
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.data_pipeline import DataFetcher, DataCleaner, FeatureEngineer, DataPreprocessor
from src.models import create_dataloaders, AlphaMLP, AlphaTrainer, AlphaPredictor
from src.risk_models import RiskModel, BetaCalculator
from src.optimization import PortfolioOptimizer
from src.backtest import BacktestEngine, PerformanceEvaluator, BacktestVisualizer

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Chuẩn bị Pipeline và Mô hình AI đã huấn luyện


In [8]:
fetcher = DataFetcher()
raw_data = fetcher.fetch_all_group()
cleaned_data = DataCleaner().clean_and_align(raw_data)
feature_data = FeatureEngineer().compute_all_features(cleaned_data)
preprocessor = DataPreprocessor()
normalized_data = preprocessor.normalize_features(feature_data)

master_df = preprocessor.prepare_tabular_dataset(normalized_data, Config.START_DATE, Config.END_DATE)
train_loader, val_loader, test_loader, test_df = create_dataloaders(master_df, batch_size=64)

model = AlphaMLP(input_dim=len(FeatureEngineer.get_feature_names()))
trainer = AlphaTrainer(model, lr=1e-3)
history = trainer.fit(train_loader, val_loader, epochs=15)

predictor = AlphaPredictor(model)
test_df_with_preds = predictor.predict_all(test_df)
for sym in normalized_data.keys():
    if sym == Config.BENCHMARK_TICKER:
        continue
    sym_preds = test_df_with_preds[test_df_with_preds['symbol'] == sym].set_index('date')['predicted_mu']
    normalized_data[sym] = normalized_data[sym].merge(sym_preds, on='date', how='left')
    normalized_data[sym]['predicted_mu'] = normalized_data[sym]['predicted_mu'].ffill().fillna(0.0)


[2026-08-02 11:14:20] [INFO] [DataFetcher]: Get group symbol ['ACB', 'ANV', 'BAF', 'BCM', 'BID', 'BMP', 'BSI', 'BSR', 'BVH', 'BWE', 'CII', 'CMG', 'CTD', 'CTG', 'CTR', 'CTS', 'DBC', 'DCM', 'DGW', 'DIG', 'DPM', 'DSE', 'DXG', 'DXS', 'EIB', 'EVF', 'FPT', 'FRT', 'FTS', 'GAS', 'GEE', 'GEX', 'GMD', 'GVR', 'HAG', 'HCM', 'HDB', 'HDC', 'HDG', 'HHV', 'HPG', 'HSG', 'HT1', 'IMP', 'KBC', 'KDC', 'KDH', 'KOS', 'LPB', 'MBB', 'MSB', 'MSN', 'MWG', 'NAB', 'NKG', 'NLG', 'NT2', 'NVL', 'OCB', 'PAN', 'PC1', 'PDR', 'PHR', 'PLX', 'PNJ', 'POW', 'PVD', 'PVT', 'REE', 'SAB', 'SBT', 'SCS', 'SHB', 'SIP', 'SJS', 'SSB', 'SSI', 'STB', 'SZC', 'TCB', 'TCH', 'TPB', 'VCB', 'VCG', 'VCI', 'VGC', 'VHC', 'VHM', 'VIB', 'VIC', 'VIX', 'VJC', 'VND', 'VNM', 'VPB', 'VPI', 'VPL', 'VRE', 'VSC', 'VTP']


INFO:DataFetcher:Get group symbol ['ACB', 'ANV', 'BAF', 'BCM', 'BID', 'BMP', 'BSI', 'BSR', 'BVH', 'BWE', 'CII', 'CMG', 'CTD', 'CTG', 'CTR', 'CTS', 'DBC', 'DCM', 'DGW', 'DIG', 'DPM', 'DSE', 'DXG', 'DXS', 'EIB', 'EVF', 'FPT', 'FRT', 'FTS', 'GAS', 'GEE', 'GEX', 'GMD', 'GVR', 'HAG', 'HCM', 'HDB', 'HDC', 'HDG', 'HHV', 'HPG', 'HSG', 'HT1', 'IMP', 'KBC', 'KDC', 'KDH', 'KOS', 'LPB', 'MBB', 'MSB', 'MSN', 'MWG', 'NAB', 'NKG', 'NLG', 'NT2', 'NVL', 'OCB', 'PAN', 'PC1', 'PDR', 'PHR', 'PLX', 'PNJ', 'POW', 'PVD', 'PVT', 'REE', 'SAB', 'SBT', 'SCS', 'SHB', 'SIP', 'SJS', 'SSB', 'SSI', 'STB', 'SZC', 'TCB', 'TCH', 'TPB', 'VCB', 'VCG', 'VCI', 'VGC', 'VHC', 'VHM', 'VIB', 'VIC', 'VIX', 'VJC', 'VND', 'VNM', 'VPB', 'VPI', 'VPL', 'VRE', 'VSC', 'VTP']


[2026-08-02 11:14:20] [INFO] [DataFetcher]: Starting data ingestion for 100 symbols (2021-01-01 -> 2025-12-30)...


INFO:DataFetcher:Starting data ingestion for 100 symbols (2021-01-01 -> 2025-12-30)...


[2026-08-02 11:14:20] [INFO] [DataFetcher]: fetching data symbol ACB


...

[2026-08-02 11:17:14] [INFO] [DataFetcher]: fetching data symbol VTP


INFO:DataFetcher:fetching data symbol VTP


[2026-08-02 11:17:14] [INFO] [DataFetcher]: Successfully loaded data for 98 symbols.


INFO:DataFetcher:Successfully loaded data for 98 symbols.


ValueError: Benchmark ticker VNINDEX missing from raw data.

In [7]:
raw_data["VNINDEX"]

KeyError: 'VNINDEX'

In [ ]:
trainer.save_model("best_w.pt")

## 2. Thực thi Backtest Engine Out-of-Sample (Tái cơ cấu Hàng tuần)


In [ ]:
engine = BacktestEngine(
    predictor=predictor,
    risk_model=RiskModel(),
    beta_calc=BetaCalculator(),
    optimizer=PortfolioOptimizer(),
    rebalance_freq='weekly',
    fee_rate=0.0020
)

backtest_res = engine.run(cleaned_data, normalized_data, start_date=Config.TEST_START_DATE, end_date=Config.END_DATE)
results_df = backtest_res['results_df']
weights_df = backtest_res['weights_df']
print('Hoàn tất Backtest Out-of-Sample!')


[2026-07-30 19:19:16] [INFO] [BacktestEngine]: Starting Out-of-Sample backtest (2023-07-01 -> 2024-06-30)...


INFO:BacktestEngine:Starting Out-of-Sample backtest (2023-07-01 -> 2024-06-30)...


[2026-07-30 19:19:41] [INFO] [BacktestEngine]: Out-of-Sample backtest completed successfully.


INFO:BacktestEngine:Out-of-Sample backtest completed successfully.


Hoàn tất Backtest Out-of-Sample!


## 3. Báo cáo Tóm tắt Hiệu suất (KPI Summary Table)


In [ ]:
evaluator = PerformanceEvaluator()
summary_df = evaluator.evaluate(results_df)
display(summary_df)


[2026-07-30 19:19:41] [INFO] [PerformanceEvaluator]: Calculating quantitative performance metrics...


INFO:PerformanceEvaluator:Calculating quantitative performance metrics...


,AI Market Neutral,VN-Index (Buy & Hold),Equal-Weight VN100
Cumulative Return,35.18%,10.65%,14.20%
Annualized Return (CAGR),35.84%,10.83%,14.45%
Annualized Volatility,10.90%,17.03%,13.62%
Sharpe Ratio,2.59,0.51,0.84
Max Drawdown (MDD),-6.28%,-17.45%,-13.56%
Daily Win Rate,56.05%,58.47%,60.48%
Information Ratio,1.01,0.00,0.28


## 4. Biểu đồ Đường cong Tài sản & Quản trị Rủi ro


In [ ]:
Config.DATA_DIR = "content/drive/MyDrive"
print(Config.DATA_DIR)

content/drive/MyDrive


In [ ]:
visualizer = BacktestVisualizer(output_dir=Path('notebook_charts'))

# 1. Equity Curves
visualizer.plot_equity_curves(results_df)
plt.show()

# 2. Drawdowns
visualizer.plot_drawdowns(results_df)
plt.show()

# 3. Rolling Beta Verification
visualizer.plot_rolling_beta(results_df)
plt.show()

# 4. Long/Short Exposure
visualizer.plot_exposure_history(weights_df)
plt.show()
